Một chiến lược mua: Khi giá dự đoán tăng so với giá hiện tại, Momentum dương (điều này cho thấy xu hướng giá đang tăng), và RSI dưới 30 (điều này cho thấy rằng cổ phiếu không bị mua quá mức).

Một chiến lược bán: Khi giá dự đoán giảm so với giá hiện tại, Momentum âm (điều này cho thấy xu hướng giá đang giảm), và RSI trên 70 (điều này cho thấy cổ phiếu không bị bán quá mức).

# Load du lieu len

In [13]:
import sys
sys.path.append('../Common')

import CommonMT5

import MetaTrader5 as mt5
from datetime import datetime, timedelta

symbol = 'ETHUSD'
from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=2)).strftime('%Y-%m-%d')

timeframe = mt5.TIMEFRAME_M15
# print(timeframe)
data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)

data

,Datetime,Open,High,Low,Close,Volume
0,2025-10-21 17:00:00,3867.43,3928.89,3866.07,3928.53,1226
1,2025-10-21 17:15:00,3928.60,3958.69,3916.53,3918.67,1218
2,2025-10-21 17:30:00,3916.66,3941.65,3916.63,3932.13,1228
3,2025-10-21 17:45:00,3932.33,4027.35,3923.33,4023.00,1258
4,2025-10-21 18:00:00,4024.00,4050.12,3999.15,4035.44,1221
...,...,...,...,...,...,...
95,2025-10-22 16:45:00,3844.03,3878.85,3833.87,3875.83,1203
96,2025-10-22 17:00:00,3874.98,3876.18,3835.08,3842.98,1251
97,2025-10-22 17:15:00,3843.55,3851.40,3789.23,3812.96,1231
98,2025-10-22 17:30:00,3813.50,3820.23,3795.26,3795.53,1175


# Tim tham so p, d, q toi uu: Tim 1 lan thoi

In [ ]:
# import itertools
# import statsmodels.api as sm
# import pandas as pd

# # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# # data.set_index('Datetime', inplace=True) da dat o tren

# # Xác định khoảng giá trị cho tham số p, d, q
# p = d = q = range(0, 6) # 0 -> 5

# pdq = list(itertools.product(p, d, q))

# best_aic = float("inf")
# best_pdq = None
# best_model = None
# data['MA'] = data['Close'].rolling(window=5).mean().dropna()  # Ví dụ dùng cửa sổ trượt kích thước 5

# for param in pdq:
#     try:
#         model = sm.tsa.ARIMA(data['MA'], order=param)
#         results = model.fit()
#         if results.aic < best_aic:
#             best_aic = results.aic
#             best_pdq = param
#             best_model = results
#     except:
#         continue

# print(f'Best ARIMA{best_pdq} AIC:{best_aic}')

# Tinh toan chien luoc

In [14]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import sys
import talib

# Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
# Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
# data.set_index('Datetime', inplace=True)

# Tính Momentum
data['Momentum'] = data['Close'].diff()

# Tính RSI
data['RSI'] = talib.RSI(data['Close'], timeperiod=14)

# Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
model = ARIMA(data['Close'], order=(1, 0, 4))
model_fit = model.fit(method_kwargs={'maxiter': 1000, 'disp': 0})  # Cập nhật số lần lặp tối đa và không hiển thị thông báo

# Dự đoán giá
data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

# Xác định tín hiệu mua và bán
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 30))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 0) & (data['RSI'] > 70))

# Trực quan hóa
import plotly.graph_objects as go

# Tạo biểu đồ cơ bản với giá đóng cửa
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))

# Thêm dữ liệu giá dự đoán vào biểu đồ
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close', line=dict(dash='dot', color='red')))

# Thêm điểm mua
buy_signals = data[data['Buy_Signal']]
fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['Close'], mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))

# Thêm điểm bán
sell_signals = data[data['Sell_Signal']]
fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['Close'], mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Tùy chỉnh layout
fig.update_layout(title='Trading Signals with ARIMA, Momentum, and RSI', xaxis_title='Date', yaxis_title='Price', hovermode='x')

# Hiển thị biểu đồ
fig.show()

In [15]:
data

,Datetime,Open,High,Low,Close,Volume,Momentum,RSI,Predicted_Close,Buy_Signal,Sell_Signal
0,2025-10-21 17:00:00,3867.43,3928.89,3866.07,3928.53,1226,NaN,NaN,3880.559509,False,False
1,2025-10-21 17:15:00,3928.60,3958.69,3916.53,3918.67,1218,-9.86,NaN,3927.416142,False,False
2,2025-10-21 17:30:00,3916.66,3941.65,3916.63,3932.13,1228,13.46,NaN,3916.761957,False,False
3,2025-10-21 17:45:00,3932.33,4027.35,3923.33,4023.00,1258,90.87,NaN,3932.584323,False,False
4,2025-10-21 18:00:00,4024.00,4050.12,3999.15,4035.44,1221,12.44,NaN,4026.408680,False,False
...,...,...,...,...,...,...,...,...,...,...,...
95,2025-10-22 16:45:00,3844.03,3878.85,3833.87,3875.83,1203,32.65,60.496947,3843.824068,False,False
96,2025-10-22 17:00:00,3874.98,3876.18,3835.08,3842.98,1251,-32.85,47.524477,3879.532948,False,False
97,2025-10-22 17:15:00,3843.55,3851.40,3789.23,3812.96,1231,-30.02,39.242943,3835.105462,False,False
98,2025-10-22 17:30:00,3813.50,3820.23,3795.26,3795.53,1175,-17.43,35.387177,3817.605848,False,False


In [ ]:
import plotly.graph_objects as go

# Giả sử DataFrame của bạn tên là `data` và có chỉ mục datetime
fig = go.Figure()

# Thêm đường biểu diễn giá đóng cửa và giá dự đoán
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Close'))
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

# Thêm điểm cho tín hiệu mua và bán
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], 
                         mode='markers', name='Buy Signal', marker=dict(color='green', size=10, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], 
                         mode='markers', name='Sell Signal', marker=dict(color='red', size=10, symbol='triangle-down')))

# Thêm đường cho Momentum và RSI với trục y phụ
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], name='Momentum', yaxis='y2'))
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], name='RSI', yaxis='y3'))

# Tùy chỉnh layout cho các trục y phụ
fig.update_layout(
    title='Trading Signals with ARIMA, Momentum, and RSI',
    xaxis_title='Date',
    yaxis_title='Close Price',
    yaxis2=dict(
        title='Momentum',
        titlefont=dict(color='purple'),
        tickfont=dict(color='purple'),
        overlaying='y',
        side='right'
    ),
    yaxis3=dict(
        title='RSI',
        titlefont=dict(color='orange'),
        tickfont=dict(color='orange'),
        overlaying='y',
        side='right',
        position=0.95
    )
)

# Hiển thị biểu đồ
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Mã chuẩn bị dữ liệu ở đây
# Giả sử 'data' là DataFrame của bạn và nó có 'Datetime' làm chỉ mục

# Tạo biểu đồ con: 3 hàng cho Giá đóng cửa, Momentum, và RSI
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Trading Signals with ARIMA, Momentum, and RSI', 'Momentum', 'RSI'),
                    vertical_spacing=0.1)

# Thêm giá đóng và dự đoán giá đóng vào hàng đầu tiên
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], mode='lines', name='Giá Đóng'), row=1, col=1)
fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Dự Đoán Giá Đóng'), row=1, col=1)

# Thêm tín hiệu mua và bán trên cùng một biểu đồ con
fig.add_trace(go.Scatter(x=data[data['Buy_Signal']].index, y=data[data['Buy_Signal']]['Close'], mode='markers',
                         marker=dict(color='green', size=10, symbol='triangle-up'), name='Tín Hiệu Mua'), row=1, col=1)
fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index, y=data[data['Sell_Signal']]['Close'], mode='markers',
                         marker=dict(color='red', size=10, symbol='triangle-down'), name='Tín Hiệu Bán'), row=1, col=1)

# Thêm Momentum vào hàng thứ hai
fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], mode='lines', name='Momentum'), row=2, col=1)

# Thêm RSI vào hàng thứ ba
fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], mode='lines', name='RSI'), row=3, col=1)

# Cập nhật layout
fig.update_layout(height=800, title_text="Chiến Lược Giao Dịch với Tín Hiệu Mua/Bán, Momentum và RSI")

# Hiển thị biểu đồ
fig.show()
